# 01. システムズシンキング（Systems Thinking） — 練習問題

**対象技術**: AI（生成AI・機械学習・自動化を含む）

技術の波及効果を「符号付き因果ループ図（causal loop diagram）」として表現し、閉ループ（フィードバックループ）を列挙して、各ループを構成するリンク符号の積から強化ループ（reinforcing, R）・均衡ループ（balancing, B）を判定する手法である。ループ構造を機械的に解析することで、少ない介入で構造全体を動かせるレバレッジ変数を見つける手がかりが得られる。

必要なライブラリを読み込む。`numpy` で符号付き隣接行列を扱い、`matplotlib` で因果ループ図を描画する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## モデル定義

生成AI普及の因果ループ図を構成する変数を列挙する。「Web上のAI生成コンテンツ蓄積量」はストック変数である。

In [ ]:
VARIABLES = [
    "生成AI利用量",            # 0
    "自動化による生産性",      # 1
    "AI投資",                  # 2
    "モデル能力",              # 3
    "AI生成コンテンツ蓄積量",  # 4 (ストック変数)
    "学習データのAI汚染度",    # 5
    "モデル出力品質",          # 6
    "人間の検証スキル",        # 7
    "偽情報の流通量",          # 8
    "AIへの社会的信頼",        # 9
]

for i, v in enumerate(VARIABLES):
    print(f"  {i}: {v}")

リンクを `(原因, 結果, 極性)` で定義する。極性 `+1` は同方向、`-1` は逆方向の影響を表す。

In [ ]:
# リンク (原因, 結果, 極性)。極性 +1: 同方向, -1: 逆方向。
EDGES = [
    # 強化ループ R1: 利用 -> 生産性 -> 投資 -> 能力 -> 利用
    (0, 1, +1),   # 利用量↑ -> 自動化による生産性↑
    (1, 2, +1),   # 生産性↑ -> AI投資↑
    (2, 3, +1),   # 投資↑ -> モデル能力↑
    (3, 0, +1),   # 能力↑ -> 利用量↑
    # 強化ループ R2: 利用 -> AI生成コンテンツ蓄積 -> 学習データ汚染
    (0, 4, +1),   # 利用量↑ -> AI生成コンテンツ蓄積↑
    (4, 5, +1),   # 蓄積↑ -> 学習データのAI汚染度↑
    (5, 3, -1),   # 汚染↑ -> モデル能力↓ (model collapse の芽)
    # 均衡ループ B1: 汚染 -> 品質低下 -> 利用抑制
    (5, 6, -1),   # 汚染↑ -> モデル出力品質↓
    (6, 0, +1),   # 出力品質↑ -> 利用量↑ (品質低下時は利用も減る)
    # 均衡ループ B2: 能力 -> 検証スキル劣化 -> 偽情報 -> 信頼低下 -> 利用抑制
    (3, 7, -1),   # モデル能力↑ -> 人間の検証スキル↓ (deskilling)
    (7, 8, -1),   # 検証スキル↑ -> 偽情報の流通量↓
    (8, 9, -1),   # 偽情報↑ -> AIへの社会的信頼↓
    (9, 0, +1),   # 信頼↑ -> 利用量↑ (信頼低下時は利用も減る)
]
print(f"変数 {len(VARIABLES)} 個 / リンク {len(EDGES)} 本")

## 解析関数

符号付き隣接行列の構築、深さ優先探索（DFS）による全単純閉路の列挙、符号積によるループ分類、負リンク本数の計数を行う関数を定義する。

In [ ]:
def build_adjacency(n, edges):
    """符号付き隣接行列を構築する。A[i, j] が i->j のリンク極性。"""
    A = np.zeros((n, n), dtype=int)
    for src, dst, sign in edges:
        A[src, dst] = sign
    return A


def find_all_cycles(A):
    """深さ優先探索で全ての単純閉路を列挙する。

    各閉路は最小ノード番号から始まる正規形にして重複を排除する。
    戻り値は (ノード列, 符号積) のリスト。
    """
    n = A.shape[0]
    cycles = []
    seen = set()

    def dfs(start, current, path, sign_product):
        for nxt in range(n):
            s = A[current, nxt]
            if s == 0:
                continue
            if nxt == start:
                full_sign = sign_product * s
                k = path.index(min(path))
                norm = tuple(path[k:] + path[:k])
                if norm not in seen:
                    seen.add(norm)
                    cycles.append((list(norm), full_sign))
            elif nxt > start and nxt not in path:
                dfs(start, nxt, path + [nxt], sign_product * s)

    for start in range(n):
        dfs(start, start, [start], 1)
    return cycles


def classify_cycle(sign_product):
    """符号積が正なら強化ループ(R)、負なら均衡ループ(B)。"""
    return "強化ループ R" if sign_product > 0 else "均衡ループ B"


def count_negative_links(A, nodes):
    """ループ内の負リンク本数を数える(分類の検算用)。"""
    neg = 0
    for i in range(len(nodes)):
        src = nodes[i]
        dst = nodes[(i + 1) % len(nodes)]
        if A[src, dst] < 0:
            neg += 1
    return neg

## 全閉路の列挙と分類

隣接行列を構築し、全ての閉ループを列挙して R / B に分類する。各ループの経路・符号積・負リンク数を表示する。

In [ ]:
n = len(VARIABLES)
A = build_adjacency(n, EDGES)
cycles = find_all_cycles(A)

print(f"検出された閉ループ: {len(cycles)} 本")
print("-" * 68)

loop_participation = np.zeros(n, dtype=int)  # 各変数のループ参加数
r_count = b_count = 0
for idx, (nodes, sign) in enumerate(cycles, start=1):
    kind = classify_cycle(sign)
    if sign > 0:
        r_count += 1
    else:
        b_count += 1
    neg = count_negative_links(A, nodes)
    names = " -> ".join(VARIABLES[v] for v in nodes)
    print(f"ループ{idx} [{kind}]")
    print(f"  経路 : {names} -> (先頭へ)")
    print(f"  符号積={sign:+d}  負リンク数={neg}"
          f"({'偶数=R' if neg % 2 == 0 else '奇数=B'})")
    for v in nodes:
        loop_participation[v] += 1
print("-" * 68)
print(f"内訳: 強化ループ R = {r_count} 本 / 均衡ループ B = {b_count} 本")

## レバレッジ候補の素朴な指標

各変数が何本のループを通過するかを集計する。多くのループを通過する変数は、そこへの介入が複数ループに同時作用するためレバレッジが高い。

In [ ]:
print("各変数のループ参加数(多いほどレバレッジ候補):")
order = np.argsort(-loop_participation)
for v in order:
    bar = "#" * loop_participation[v]
    print(f"  {VARIABLES[v]:<22} {loop_participation[v]:>2} {bar}")

top = VARIABLES[order[0]]
print()
print("[解釈]")
print(f"  最も多くのループを通過する変数は『{top}』。")
print("  ここへの介入は複数ループに同時作用するためレバレッジが高い。")
print("  AI生成コンテンツの蓄積は model collapse とスキル空洞化という")
print("  意図せざる結果を同時に強める構造点である。")

## 可視化: 因果ループ図

変数ノードを円周上に配置し、リンクを矢印で描く。極性 + は実線・暖色、− は破線・寒色で色分けし、検出した強化/均衡ループの本数を注記する。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

# 変数ノードを円周上に配置
angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, n, endpoint=False)
pos = np.column_stack([np.cos(angles), np.sin(angles)])

# リンク(矢印)を描画: + は実線/暖色, - は破線/寒色
for src, dst, sign in EDGES:
    x0, y0 = pos[src]
    x1, y1 = pos[dst]
    color = "#d1495b" if sign > 0 else "#3a6ea5"
    style = "-" if sign > 0 else "--"
    ax.annotate(
        "", xy=(x1 * 0.82, y1 * 0.82), xytext=(x0 * 0.82, y0 * 0.82),
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.8,
                        linestyle=style, shrinkA=14, shrinkB=14,
                        connectionstyle="arc3,rad=0.15"))

# ノードを描画
for i, (x, y) in enumerate(pos):
    ax.scatter(x, y, s=900, c="#f4e285", edgecolors="#5a5a5a", zorder=3)
    ax.text(x, y, f"V{i}", ha="center", va="center",
            fontsize=10, fontweight="bold", zorder=4)
    lx, ly = x * 1.32, y * 1.32
    ax.text(lx, ly, f"V{i}", ha="center", va="center", fontsize=8,
            color="#333333")

# 凡例
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], color="#d1495b", lw=2, linestyle="-",
           label="positive link (+)"),
    Line2D([0], [0], color="#3a6ea5", lw=2, linestyle="--",
           label="negative link (-)"),
]
ax.legend(handles=legend_elems, loc="upper right", fontsize=9)

ax.set_title(f"Causal Loop Diagram (AI)\n"
             f"Reinforcing R={r_count}, Balancing B={b_count}",
             fontsize=12)
ax.set_xlim(-1.6, 1.6)
ax.set_ylim(-1.6, 1.6)
ax.set_aspect("equal")
ax.axis("off")

# 変数IDの対応をテキストで添える
note = "\n".join(f"V{i}: {VARIABLES[i]}" for i in range(n))
ax.text(-1.55, -1.55, note, fontsize=7, va="bottom", ha="left")

plt.tight_layout()
plt.show()

## ループが生む動的な振る舞い (behavior-over-time)

ここまでは「どんなループが存在するか」という静的な構造分析だった。だがシステムズシンキングの分析的中核は「そのループが時間とともに何を起こすか」を読むことにある。符号付き隣接行列に簡易な離散時間ダイナミクスを与え、各変数を [0,1] に正規化したロジスティック型の更新式で時間推移を計算する。強化ループは増殖して飽和へ、均衡ループは抑制として現れる。

In [ ]:
def simulate(A, x0, gain=0.9, dt=0.25, steps=80):
    """符号付き隣接行列 A に離散時間ダイナミクスを与えて時間推移を返す。

    net_i(t) = sum_j A[j,i] * gain * x_j(t)
    x_i(t+1) = x_i(t) + dt * ( net_i*(1-x_i)  if net_i > 0
                               net_i*x_i      if net_i <= 0 )
    変数を [0,1] に正規化し、ロジスティック型の飽和を持たせる。
    強化ループ -> 増殖して飽和、均衡ループ -> 抑制、として振る舞う。
    """
    n = A.shape[0]
    x = np.array(x0, dtype=float)
    traj = np.zeros((steps + 1, n))
    traj[0] = x
    for t in range(steps):
        net = (A.astype(float) * gain).T @ x   # net[i] = sum_j A[j,i]*gain*x[j]
        dx = np.where(net > 0, net * (1.0 - x), net * x)
        x = np.clip(x + dt * dx, 0.0, 1.0)
        traj[t + 1] = x
    return traj


# 初期値: 注目変数と「同じ強化ループ上の変数」に成長の種を与え、
# 残りは低い水準から開始する。強化ループに点火し、均衡ループが
# あとから効いてくる — という典型的な立ち上がりを再現するため。
FOCUS = 0        # 注目する成果変数のインデックス
seed_vars = {FOCUS}
for nodes, sign in cycles:
    if sign > 0 and FOCUS in nodes:      # focus を含む強化ループ
        seed_vars |= set(nodes)
x0 = np.full(n, 0.10)
for v in seed_vars:
    x0[v] = 0.25
x0[FOCUS] = 0.15
traj = simulate(A, x0, gain=0.9, dt=0.25, steps=80)
print(f"シミュレーション完了: {traj.shape[0]} ステップ x {traj.shape[1]} 変数")
print(f"注目変数『{VARIABLES[FOCUS]}』 "
      f"初期={traj[0, FOCUS]:.3f} -> 最終={traj[-1, FOCUS]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
cmap = plt.get_cmap("tab10")
for i in range(n):
    lw = 3.0 if i == FOCUS else 1.4
    alpha = 1.0 if i == FOCUS else 0.7
    ax.plot(traj[:, i], color=cmap(i % 10), lw=lw, alpha=alpha,
            label=f"V{i}: {VARIABLES[i]}")

ax.set_title("Behavior over time: how the loops actually behave", fontsize=12)
ax.set_xlabel("time step")
ax.set_ylabel("normalized level [0, 1]")
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.show()

print("[解釈]")
print("  ループが『ある』ことではなく『何を起こすか』を見るのが狙い。")
print("  強化ループに乗った変数は序盤に立ち上がり、やがて飽和する。")
print("  均衡ループに支配される変数は抑制され、低い水準に落ち着くか減衰する。")

## ループ・ドミナンスの交代 (loop dominance shift)

S字状に飽和する曲線は、単一のループではなく「支配的なループの交代」が生む。注目変数のダイナミクスを、強化ループ由来の押し上げ（成長余地 1-x で重みづけ）と、均衡ループ由来＋飽和そのものによる抑え込み（飽和度 x で重みづけ）に分解する。初期は押し上げが支配して伸び、ある時点で抑え込みが上回って飽和・減衰へ向かう。寄与が逆転する点（dominance handover）をグラフに注記する。

In [ ]:
def loop_contributions(A, traj, cycles, focus, gain=0.9):
    """注目変数のダイナミクスを、強化的な押し上げと均衡的な抑え込みに分解する。

    更新式 x(t+1) = x + dt*( net*(1-x) [net>0] / net*x [net<=0] ) に従うと、
    注目変数に入る net flux は強化ループ由来の押し上げと均衡ループ由来の
    引き下げの和になる。focus に入る各リンク j->focus の瞬間 flux を、
    そのリンクが属するループの符号で R 群 / B 群に振り分け、さらに
      - R 群（押し上げ）には成長余地 (1-x) を掛ける
      - B 群（抑え込み）には飽和度 x を掛ける
    R 寄与は飽和に近づくと自ら衰え、B 寄与は x とともに伸びる。
    両者が逆転する点が支配ループの交代（dominance handover）である。
    """
    x = traj[:, focus]
    r_flux = np.zeros(traj.shape[0])
    b_flux = np.zeros(traj.shape[0])
    for j in range(A.shape[0]):
        if A[j, focus] == 0:
            continue
        # リンク j->focus を含むループの R/B 構成を調べる
        r_loops = b_loops = 0
        for nodes, sign in cycles:
            L = len(nodes)
            if any(nodes[k] == j and nodes[(k + 1) % L] == focus
                   for k in range(L)):
                if sign > 0:
                    r_loops += 1
                else:
                    b_loops += 1
        tot = r_loops + b_loops
        if tot == 0:
            continue
        flux = abs(A[j, focus]) * gain * traj[:, j]   # 瞬間 flux の大きさ
        r_flux += flux * (r_loops / tot)
        b_flux += flux * (b_loops / tot)
    # R は成長余地で重みづけ（飽和で衰える）、B は飽和度で重みづけ（飽和で伸びる）
    r_contrib = r_flux * (1.0 - x)
    b_contrib = b_flux * x + r_flux * x   # 飽和そのものも均衡的な抑え込みに数える
    return r_contrib, b_contrib


r_c, b_c = loop_contributions(A, traj, cycles, FOCUS, gain=0.9)

# R 寄与と B 寄与の大小が逆転する handover 時刻を検出
handover = None
for t in range(1, len(r_c)):
    if r_c[t - 1] >= b_c[t - 1] and r_c[t] < b_c[t]:
        handover = t
        break
print(f"強化的な押し上げ寄与の最大値 : {r_c.max():.4f}")
print(f"均衡的な抑え込み寄与の最大値 : {b_c.max():.4f}")
print(f"ドミナンス交代時刻           : {handover}" if handover is not None
      else "交代点は検出されず（強化ループが終始支配）")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})

# 上段: 注目変数の時間推移
ax1.plot(traj[:, FOCUS], color="#222222", lw=2.6,
         label=f"V{FOCUS}: {VARIABLES[FOCUS]}")
ax1.set_ylabel("normalized level")
ax1.set_title(f"Loop dominance shift on V{FOCUS} ({VARIABLES[FOCUS]})",
              fontsize=12)
ax1.grid(alpha=0.3)
ax1.set_ylim(-0.05, 1.05)

# 下段: 押し上げ寄与 / 抑え込み寄与
ax2.plot(r_c, color="#d1495b", lw=2.0,
         label="reinforcing push (R)")
ax2.plot(b_c, color="#3a6ea5", lw=2.0,
         label="balancing + saturation hold-down (B)")
steps_x = range(len(r_c))
ax2.fill_between(steps_x, r_c, b_c, where=(r_c >= b_c),
                 color="#d1495b", alpha=0.15, label="R dominates")
ax2.fill_between(steps_x, r_c, b_c, where=(b_c > r_c),
                 color="#3a6ea5", alpha=0.15, label="B dominates")
ax2.set_xlabel("time step")
ax2.set_ylabel("loop contribution")
ax2.grid(alpha=0.3)

if handover is not None:
    for ax in (ax1, ax2):
        ax.axvline(handover, color="#5a5a5a", ls="--", lw=1.5)
    ax1.annotate("dominance handover\n(R -> B)",
                 xy=(handover, traj[handover, FOCUS]),
                 xytext=(handover + 6, traj[handover, FOCUS] - 0.28),
                 fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="#5a5a5a"))

ax1.legend(fontsize=8, loc="lower right")
ax2.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

print("[解釈]")
print("  序盤は強化ループの寄与が大きく、注目変数は加速度的に伸びる。")
print("  破線の時刻で均衡ループの寄与が上回り、成長が頭打ち・減衰へ転じる。")
print("  S字曲線の屈曲は単一ループではなく『支配ループの交代』が生んでいる。")

## レバレッジポイントの特定 (sensitivity analysis)

ドネラ・メドウズは、システムには「小さな介入が大きな構造変化を生む点（レバレッジポイント）」が存在すると論じた。ここでは各リンク重みを微小に変動させて再シミュレーションし、注目する成果変数の最終値の変化量を感度として測る。感度が大きいリンクほど、最小の介入で最大の変化を生むレバレッジポイントである。

In [ ]:
def link_sensitivities(edges, n, focus, x0, delta=0.15,
                       gain=0.9, dt=0.25, steps=80):
    """各リンク重みを delta だけ微小変動させ、注目変数の最終値の変化量を測る。

    重みを (1+delta) 倍したときの最終値 - (1-delta) 倍したときの最終値、
    の絶対値を感度（影響度）とする中心差分の感度分析。
    """
    sens = []
    for k, (src, dst, sign) in enumerate(edges):
        finals = []
        for factor in (1.0 - delta, 1.0 + delta):
            A_mod = np.zeros((n, n))
            for s2, d2, sg2 in edges:
                A_mod[s2, d2] = sg2
            A_mod[src, dst] = sign * factor
            tr = simulate(A_mod, x0, gain=gain, dt=dt, steps=steps)
            finals.append(tr[-1, focus])
        sens.append(abs(finals[1] - finals[0]))
    return np.array(sens)


sens = link_sensitivities(EDGES, n, FOCUS, x0)
ranking = np.argsort(-sens)
print(f"レバレッジ・ランキング（注目変数『{VARIABLES[FOCUS]}』の最終値への感度）")
print("-" * 60)
for rank, k in enumerate(ranking, start=1):
    src, dst, sign = EDGES[k]
    print(f"  {rank:>2}. V{src}->V{dst} ({'+' if sign > 0 else '-'})"
          f"  {VARIABLES[src]} -> {VARIABLES[dst]}")
    print(f"      感度 = {sens[k]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6))
labels = [f"V{EDGES[k][0]}->V{EDGES[k][1]}" for k in ranking]
vals = sens[ranking]
colors = ["#e07a3f" if i < 3 else "#9fb8c8" for i in range(len(vals))]
ax.barh(range(len(vals)), vals, color=colors)
ax.set_yticks(range(len(vals)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("sensitivity = |change of focus-variable final level|")
ax.set_title("Leverage points: links ranked by sensitivity", fontsize=12)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

top_k = ranking[0]
ts, td, _ = EDGES[top_k]
print("[解釈]")
print(f"  最も感度の高いリンクは V{ts}->V{td}"
      f"（{VARIABLES[ts]} -> {VARIABLES[td]}）。")
print("  このリンクの強さを少し変えるだけで成果変数の到達点が大きく動く。")
print("  メドウズの言うレバレッジポイント — 最小の介入で最大の変化を生む点 —")
print("  であり、限られた政策資源を投じるべき構造点を示している。")

## システム原型の検出 (system archetypes)

システム原型は、異なる文脈で繰り返し現れるループ構造のパターンである。代表が「成長の限界（limits to growth）」——強化ループによる成長が、ある制約ループによって必ず頭打ちになる構造。ここでは検出した全ループから、強化ループと均衡ループが少なくとも1つの変数を共有する組を探し、limits-to-growth 原型として列挙する。

In [ ]:
def detect_limits_to_growth(cycles):
    """強化ループと均衡ループが変数を共有する組を limits-to-growth として検出。"""
    r_loops = [(i, set(nd)) for i, (nd, sg) in enumerate(cycles) if sg > 0]
    b_loops = [(i, set(nd)) for i, (nd, sg) in enumerate(cycles) if sg <= 0]
    found = []
    for ri, rset in r_loops:
        for bi, bset in b_loops:
            shared = rset & bset
            if shared:
                found.append((ri, bi, sorted(shared)))
    return found


def detect_shifting_the_burden(cycles):
    """参考: 2 本以上の均衡ループが変数を共有する『問題のすり替え』の簡易判定。"""
    b_loops = [(i, set(nd)) for i, (nd, sg) in enumerate(cycles) if sg <= 0]
    found = []
    for a in range(len(b_loops)):
        for b in range(a + 1, len(b_loops)):
            shared = b_loops[a][1] & b_loops[b][1]
            if shared:
                found.append((b_loops[a][0], b_loops[b][0], sorted(shared)))
    return found


ltg = detect_limits_to_growth(cycles)
print(f"検出された『成長の限界 (limits to growth)』原型: {len(ltg)} 組")
print("=" * 64)
for ri, bi, shared in ltg:
    r_nodes, _ = cycles[ri]
    b_nodes, _ = cycles[bi]
    print(f"  強化ループ: {' -> '.join('V%d' % v for v in r_nodes)} -> (先頭へ)")
    print(f"  均衡ループ: {' -> '.join('V%d' % v for v in b_nodes)} -> (先頭へ)")
    print(f"  共有変数  : {', '.join(VARIABLES[v] for v in shared)}")
    print("-" * 64)

stb = detect_shifting_the_burden(cycles)
print(f"参考 — 均衡ループ同士が変数を共有する組（問題のすり替えの芽）: {len(stb)} 組")

In [ ]:
print("[解釈] システム原型としての読み")
print()
print("AI 題材では『生成AI利用量↑ → AI生成コンテンツ蓄積↑ → 学習データ汚染↑ → モデル能力↓/出力品質↓ → 利用量↓』が典型的な『成長の限界』である。前半は R1（利用→生産性→投資→能力→利用）が利用を増殖させるが、蓄積というストックが汚染を通じて均衡ループを呼び覚まし、成長を頭打ちにする。")
print()
if ltg:
    ri, bi, shared = ltg[0]
    sv = VARIABLES[shared[0]]
    print(f"  検出された原型の核となる共有変数は『{sv}』。")
    print("  強化ループが成長を駆動する一方、同じ変数を通る均衡ループが")
    print("  制約として働き、成長は必ず頭打ちになる。")
    print("  対策の定石は『成長を速める』ことではなく『制約ループを緩める』こと —")
    print("  すなわちレバレッジポイント分析(C)で上位に来たリンクへの介入である。")
else:
    print("  このモデルでは limits-to-growth 原型は検出されなかった。")

## 未来デザイン論文での使われ方と結論への影響

システムズシンキングは、未来デザイン論文において「何が起きるか」を当てる予測装置としてではなく、「どこを動かせば望ましい方向へ系を導けるか」を論じる介入設計の装置として用いられる。論文は典型的に、対象技術をめぐる主要変数を因果ループ図に編成し、強化ループと均衡ループ、そしてリンクに伴う遅延を提示する。そのうえで、なぜ単純な線形外挿が誤るのか——フィードバックが効果を増幅または減衰させ、遅延が原因と結果を時間的に切り離すから——を構造的に説明し、結論を「予測値の提示」から「レバレッジポイントの特定」へと組み替える。読者に手渡されるのは将来の数値ではなく、最小の介入で最大の構造変化を生む地点の地図である。

この手法がもたらす結論の型は、おのずと「どこを動かすか」という規範的・処方的な形をとる。境界設定の経路で見れば、結論はモデルに取り込んだ変数とループの内部でのみ成立し、図に描かれなかった要素は最初から論証の射程外に置かれる。時間観の経路では、未来は内生的な構造から生成されるものと捉えられ、過去の延長でも単なる選択対象でもなく「現在のループ構造がほどけていく先」として描かれる。価値の所在は、何を変数とし何をレバレッジと呼ぶかという構造の切り取り方そのものに埋め込まれる。

同時に、この手法は固有のバイアスを結論に持ち込む。内生的なフィードバック構造を重視するため、モデル境界の外から来る外生ショック——突発的な規制転換、地政学的断絶、無関係な技術の波及——は構造的に過小評価されやすい。また、ループの存在を強調する論の運びは、意図せざる結果や政策抵抗（介入が均衡ループを刺激して打ち消される現象）を前景化させる方向に結論を傾け、「素朴な介入はうまくいかない」という慎重論へ自然に着地しがちである。論文がこの手法を採るとき、結論の説得力はモデル境界の妥当性と外生要因の扱いをどれだけ誠実に明示するかにかかっている。

## 発展課題

**課題A**: `simulate` の `gain` や `dt`、初期値 `x0` を変えて behavior-over-time の形（飽和の速さ、オーバーシュートの有無）がどう変わるか観察せよ。さらに `loop_contributions` を使い、ドミナンス交代の時刻が初期条件によってどう前後するかを調べ、強化ループと均衡ループのどちらを「先に・強く」回すかが結果を左右することを確認せよ。

**課題B**: 自分で新しい変数とリンクを `EDGES` に追加し（題材に応じて成長の限界を緩める制約緩和ループや、遅延を伴う独自ループを設計する）、(C) のレバレッジ・ランキングと (D) の原型検出がどう変わるかを観察せよ。追加したリンクが上位レバレッジに食い込むか、新たな limits-to-growth 原型が生まれるかを論じること。